In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/aquadata/ClymeneDolphin/ClymeneDolphin/8300600H.wav
/kaggle/input/aquadata/ClymeneDolphin/ClymeneDolphin/83035022.wav
/kaggle/input/aquadata/ClymeneDolphin/ClymeneDolphin/8303502Z.wav
/kaggle/input/aquadata/ClymeneDolphin/ClymeneDolphin/8303501X.wav
/kaggle/input/aquadata/ClymeneDolphin/ClymeneDolphin/83035034.wav
/kaggle/input/aquadata/ClymeneDolphin/ClymeneDolphin/8300602P.wav
/kaggle/input/aquadata/ClymeneDolphin/ClymeneDolphin/83006042.wav
/kaggle/input/aquadata/ClymeneDolphin/ClymeneDolphin/8300603M.wav
/kaggle/input/aquadata/ClymeneDolphin/ClymeneDolphin/8303501Y.wav
/kaggle/input/aquadata/ClymeneDolphin/ClymeneDolphin/8303501M.wav
/kaggle/input/aquadata/ClymeneDolphin/ClymeneDolphin/8300603Y.wav
/kaggle/input/aquadata/ClymeneDolphin/ClymeneDolphin/8300600U.wav
/kaggle/input/aquadata/ClymeneDolphin/ClymeneDolphin/83006023.wav
/kaggle/input/aquadata/ClymeneDolphin/ClymeneDolphin/8303503X.wav
/kaggle/input/aquadata/ClymeneDolphin/ClymeneDolphin/8300603V.wav
/kaggle/in

In [2]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

2025-04-27 16:09:34.755785: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745770175.024142      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745770175.094378      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Num GPUs Available:  2


In [2]:
import torch
print(torch.cuda.is_available())

True


In [3]:
import numpy as np
import pandas as pd
import librosa
import os
import torch
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, GRU, Dense, Dropout, Flatten, MaxPooling1D, BatchNormalization
from tensorflow.keras.activations import swish
from tensorflow.keras.callbacks import ReduceLROnPlateau

2025-04-28 04:10:51.012255: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745813451.205517      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745813451.262069      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
# Load Wav2Vec2.0 model
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/configuration_utils.py:311: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

In [5]:
import os
import librosa
import numpy as np

# Set directories
main_dir = "/kaggle/input/aquadata"
labels = os.listdir(main_dir)
x, y = [], []

In [6]:
%%time
for label in labels:
    sub_dir = os.path.join(main_dir, label)
    for root, dirs, files in os.walk(sub_dir):
        for file in files:
            path = os.path.join(root, file)
            
            if not file.lower().endswith(('.wav', '.mp3', '.flac')):  # Check only audio files
                continue

            audio, sr = librosa.load(path, sr=16000)

            # Ensure minimum length for FFT
            min_length = 2048
            if len(audio) < min_length:
                audio = np.pad(audio, (0, min_length - len(audio)), mode='constant')

            # Dynamic FFT size
            n_fft = min(2048, len(audio) // 2)
            hop_length = n_fft // 4

            # Extract MFCCs
            mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=80, n_fft=n_fft, hop_length=hop_length)
            delta_mfcc = librosa.feature.delta(mfccs)
            delta2_mfcc = librosa.feature.delta(mfccs, order=2)

            features = np.vstack([mfccs, delta_mfcc, delta2_mfcc])
            mfccs_scaled = np.mean(features.T, axis=0)

            x.append(mfccs_scaled)
            y.append(label)

CPU times: user 8min 48s, sys: 23.3 s, total: 9min 11s
Wall time: 7min 26s


In [18]:
# Encode Labels
le = LabelEncoder().fit(y)
y = le.transform(y)
np.save("APR27.npy", le.classes_)

# Split Data
x_train, x_test, y_train, y_test = train_test_split(np.array(x), np.array(y), test_size=0.2, random_state=42)
x_train = np.expand_dims(x_train, axis=2)
x_test = np.expand_dims(x_test, axis=2)

In [36]:
from tensorflow.keras.layers import GaussianNoise

# Apply Gaussian Noise to the input to improve generalization
model = Sequential()

# Convolutional Layers
model.add(Conv1D(128, 5, padding='same', input_shape=(x_train.shape[1], 1), kernel_regularizer=tf.keras.regularizers.l2(0.0005)))
model.add(BatchNormalization())
model.add(tf.keras.layers.LeakyReLU(alpha=0.01))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.3))  # Increased dropout

model.add(Conv1D(256, 5, padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.0005)))
model.add(BatchNormalization())
model.add(tf.keras.layers.LeakyReLU(alpha=0.01))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.4))  # Increased dropout

model.add(Conv1D(384, 3, padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.0005)))  # Reduced filters slightly
model.add(BatchNormalization())
model.add(tf.keras.layers.LeakyReLU(alpha=0.01))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.5))  # Increased dropout

# GRU Layers
model.add(GRU(192, activation='tanh', return_sequences=True, kernel_regularizer=tf.keras.regularizers.l2(0.0005)))  # Reduced units slightly
model.add(GRU(128, activation='tanh', return_sequences=True, kernel_regularizer=tf.keras.regularizers.l2(0.0005)))
model.add(GRU(96, activation='tanh', kernel_regularizer=tf.keras.regularizers.l2(0.0005)))
model.add(Dropout(0.5))  # Increased dropout

# Fully Connected Layers
model.add(Dense(512, activation='swish', kernel_regularizer=tf.keras.regularizers.l2(0.0005)))  # L2 Regularization added
model.add(Dropout(0.6))
model.add(Dense(len(labels), activation='softmax'))


CPU times: user 260 ms, sys: 11.3 ms, total: 271 ms
Wall time: 260 ms


In [30]:
# Evaluate Model
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9409 - loss: 0.4450
Test Accuracy: 93.50%
Test Loss: 0.4569


In [32]:
# Save Model
model.save("93.50.keras")

In [31]:

# Evaluate model
model.evaluate(x_test, y_test)

# Training accuracy and loss over epochs
acc = history.history['accuracy'] # Changed detector to history
val_acc = history.history['val_accuracy'] # Changed detector to history
loss = history.history['loss'] # Changed detector to history
val_loss = history.history['val_loss'] # Changed detector to history

# Print final accuracy and loss values
print(f"Final Training Accuracy: {acc[-1] * 100:.2f}%")
print(f"Final Validation Accuracy: {val_acc[-1] * 100:.2f}%")
print(f"Final Training Loss: {loss[-1]:.4f}")
print(f"Final Validation Loss: {val_loss[-1]:.4f}")


95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9409 - loss: 0.4450
Final Training Accuracy: 96.79%
Final Validation Accuracy: 93.50%
Final Training Loss: 0.3006
Final Validation Loss: 0.4569
